This notebook reproduces the paper's Fig. 4/Fig. 5 comparison of "full" (all singular values comparable, high effective dimension) vs. "cutoff" (correlation spectrum truncated after `cutoff` and decayed via `decay_exp`, low effective dimension) dense, 1-feature Fourier regression models, for three data-generating regimes: fully biased (the data lies exactly in both models' expressible space at a reference `theta0`), unbiased (the data is unrelated to the models), and slightly biased (the data interpolates between the two as the perturbation strength `eps` = `delta_data` is scanned). It loads the pre-aggregated results produced by `extract_data_for_plots.py` from `train_and_FIM_biased.py`, `train_and_FIM_UNbiased.py` and `train_and_FIM_SLIGHTbiased.py`, then plots MSE training curves for the full vs. cutoff models and `Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min` against both the effective-dimension gap (`d_eff^full - d_eff^cut`) and the partial-bias parameter `delta_data`. The central result reproduced here is that `Delta_{f-c}MSE_min` is positive for biased models (low ED trains better) and negative for unbiased models (high ED trains better), with the slightly-biased case interpolating smoothly between the two as `delta_data` is varied.

In [ ]:
# Importing necessary packages
import sys
import os
import copy
import importlib
import pickle

import pennylane.numpy as np



import matplotlib
import matplotlib.pyplot as plt



# Current path for importing custom functions
path_base = '/home/b/b309245/cloud_cover_qml_parametrization/DYAMOND_cellbased_implementations/'
sys.path.insert(0, path_base + 'qml_useful_functions')

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the basic model/experiment specs (`no_of_features`, `max_frequency`, `no_params`, `dim_basis_single_param`, `cutoff`, `decay_exp_vec`, `eps_vec`, `learning_rate`, `batch_size`) used to identify and load the matching result files saved by the training scripts.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ------------------------------------ Basic model specs ----------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

# Folder in which to load data from
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/train_and_FIM_Dloc3/'

### 'biased_data_gen': the data generating funct. is fully encompassed by both full and cutoff models
### 'UNbiased_data_gen': the data generating funct. is random, not encompassed by full and cutoff models
### 'SLIGHTbiased_data_gen': the data generating funct. is only approximately encompassed by full and cutoff models

name_data_gen_fullbiased = 'biased_data_gen'
name_data_gen_unbiased = 'UNbiased_data_gen'
name_data_gen = 'SLIGHTbiased_data_gen'

eps_vec = [0.0001, 0.0002, 0.0004, 0.0007,
           0.001, 0.002, 0.004, 0.0053, 0.007, 0.0083,
           0.01, 0.0125, 0.0165, 0.02, 0.025, 0.03, 0.035, 0.04, 0.05, 0.06, 0.07, 0.08,
           0.1, 0.2, 0.3, 0.4, 0.6, 0.8,
           1.0, 2.0, 3.0, 4.0]
name_eps_vec = ['0p0001', '0p0002', '0p0004', '0p0007',
                '0p001', '0p002', '0p004', '0p0053', '0p007', '0p0083',
                '0p01', '0p0125', '0p0165', '0p02', '0p025', '0p03', '0p035', '0p04', '0p05', '0p06', '0p07', '0p08',
                '0p1', '0p2', '0p3', '0p4', '0p6', '0p8',
                '1p0', '2p0', '3p0', '4p0']

# Learning rate
learning_rate = 0.025
learning_rate_name = '0p025'

# Batch size for training
batch_size = 5
batch_size_name = str(batch_size)

### Cutoff index for correlations among Fourier components
cutoff = 6
cutoff_name = str(cutoff)

### Decay exponents for cutoff model
decay_exp_vec = [0.05, 0.07, 0.1, 0.2, 0.333, 0.5, 0.7, 1.0, 2.0, 3.33]
decay_exp_name_vec = ['0p05', '0p07', '0p1', '0p2', '0p333', '0p5', '0p7', '1p0', '2p0', '3p33']

### No. of features (dimension of input vectors)
no_of_features = 1
name_no_features = str(no_of_features)

### Maximal Fourier frequency
max_frequency = 8
name_max_freq = str(max_frequency)

### No. of parameters
no_params = 7
name_no_params = str(no_params)

### No. basis states per parameter
dim_basis_single_param = 3  ### (1, cos(th), sin(th))
name_dim_basis_param = str(dim_basis_single_param)

Estimates a normalization constant `norm_y` as the average largest singular value of many random `dim_basis_inputs x dim_basis_params` coefficient matrices, used below to rescale MSE values onto a comparable, roughly O(1) scale.

In [ ]:
local_dim_basis_inputs = 2 * max_frequency - 1
dim_basis_inputs = local_dim_basis_inputs ** no_of_features
dim_basis_params = dim_basis_single_param ** no_params
largest_Sval_all = []
for i in range(50):
    coeffs_vectors_0 = (2.0 * np.random.rand(dim_basis_inputs, dim_basis_params) - 1.0)
    U, S, Vh = np.linalg.svd(coeffs_vectors_0, full_matrices=True)
    largest_Sval_all.append(np.max(S))
largest_Sval_all = np.asarray(largest_Sval_all)
norm_y = np.mean(largest_Sval_all)
print(np.mean(largest_Sval_all))
print(np.std(largest_Sval_all))
print(np.min(largest_Sval_all))
print(np.max(largest_Sval_all))

Loads the three pre-aggregated pickle files (`extracted_results_SLIGHTbiased`/`FULLbiased`/`UNbiased`) produced by `extract_data_for_plots.py`, each keyed by `decay_exp` (or `eps`, for the slightly-biased dict) and holding per-model-draw lists of MSE, FIM-spectrum and effective-dimension statistics.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ---------------------------------------- Load data --------------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

name_end_extr = ('_Nfeatures' + name_no_features + '_MaxFreq' + name_max_freq + '_Nparams' + name_no_params + 
                 '_DimBasisParam' + name_dim_basis_param + '_cutoff' + cutoff_name)

filename = 'extracted_results_SLIGHTbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'rb') as f:
    dict_slightbiased_all = pickle.load(f)
    
filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'rb') as f:
    dict_fullbiased_all = pickle.load(f)

filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
path_file = os.path.join(results_folder, filename)
with open(path_file, 'rb') as f:
    dict_unbiased_all = pickle.load(f)

Quick sanity checks of the loaded data structure: lists the fields stored for one `decay_exp` value and checks the array shapes of `MSE_full_mean` and `norm_eff_dim_full`.

In [ ]:
for nnn in dict_fullbiased_all['1p0']:
    print(nnn)

In [ ]:
np.asarray(dict_fullbiased_all['1p0']['MSE_full_mean']).shape

In [ ]:
np.asarray(dict_fullbiased_all['1p0']['norm_eff_dim_full']).shape

MSE training curves (mean, with min/max shading) for the fully-biased full vs. cutoff model, for one `decay_exp`/model-draw combination, each labeled with its estimated normalized effective dimension `d_eff`.

In [ ]:
n_draw = 10
dec_exp_name = '2p0'
dict_data = dict_fullbiased_all

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = dict_data[dec_exp_name]['norm_eff_dim_full'][n_draw]
y_f = dict_data[dec_exp_name]['MSE_full_mean'][n_draw] / norm_y**2.0
min_y_f = dict_data[dec_exp_name]['MSE_full_min'][n_draw] / norm_y**2.0
max_y_f = dict_data[dec_exp_name]['MSE_full_max'][n_draw] / norm_y**2.0

ned_c = dict_data[dec_exp_name]['norm_eff_dim_cut'][n_draw]
y_c = dict_data[dec_exp_name]['MSE_cut_mean'][n_draw] / norm_y**2.0
min_y_c = dict_data[dec_exp_name]['MSE_cut_min'][n_draw] / norm_y**2.0
max_y_c = dict_data[dec_exp_name]['MSE_cut_max'][n_draw] / norm_y**2.0

x_f = np.arange(1, len(y_f) + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
fig.savefig('MSE_training_fullbiased_full_and_cut.png', bbox_inches='tight')
fig.savefig('MSE_training_fullbiased_full_and_cut.pdf', bbox_inches='tight')

Same MSE training-curve comparison (full vs. cutoff, mean with min/max shading, labeled by `d_eff`) but for the unbiased data-generating regime.

In [ ]:
n_draw = 0
dec_exp_name = '2p0'
dict_data = dict_unbiased_all

fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

ned_f = dict_data[dec_exp_name]['norm_eff_dim_full'][n_draw]
y_f = dict_data[dec_exp_name]['MSE_full_mean'][n_draw] / norm_y**2.0
min_y_f = dict_data[dec_exp_name]['MSE_full_min'][n_draw] / norm_y**2.0
max_y_f = dict_data[dec_exp_name]['MSE_full_max'][n_draw] / norm_y**2.0

ned_c = dict_data[dec_exp_name]['norm_eff_dim_cut'][n_draw]
y_c = dict_data[dec_exp_name]['MSE_cut_mean'][n_draw] / norm_y**2.0
min_y_c = dict_data[dec_exp_name]['MSE_cut_min'][n_draw] / norm_y**2.0
max_y_c = dict_data[dec_exp_name]['MSE_cut_max'][n_draw] / norm_y**2.0

x_f = np.arange(1, len(y_f) + 1)
ax.plot(x_f, y_f, 'o-', color='tab:blue', markersize=7, linewidth=3, label=r'$\mathrm{full:\;}\hat{d}_{\mathrm{eff}}='+str(ned_f)[:5]+'$')
ax.fill_between(x_f, min_y_f, max_y_f, alpha=0.3, edgecolor='tab:blue', facecolor='tab:blue', linewidth=0.5)

x_c = np.arange(1, len(y_c) + 1)
ax.plot(x_c, y_c, 'o-', color='tab:orange', markersize=7, linewidth=3, label=r'$\mathrm{cut:\;}\hat{d}_{\mathrm{eff}}='+str(ned_c)[:5]+'$')
ax.fill_between(x_c, min_y_c, max_y_c, alpha=0.3, edgecolor='tab:orange', facecolor='tab:orange', linewidth=0.5)

#ax.legend(fontsize=22)
ax.legend(loc='lower left', fontsize=26)
ax.set_xlabel(r'$\mathrm{epoch}$', fontsize=fs)
ax.set_ylabel(r'$\mathrm{MSE}$', fontsize=fs)
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
fig.savefig('MSE_training_unbiased_full_and_cut.png', bbox_inches='tight')
fig.savefig('MSE_training_unbiased_full_and_cut.pdf', bbox_inches='tight')

Scatter of `Delta_{f-c}MSE_min` against the effective-dimension gap `d_eff^full - d_eff^cut`, pooling all `decay_exp` values and model draws, with fully-biased points and unbiased points overlaid in different colors: biased models show `Delta_{f-c}MSE_min > 0` while unbiased models show it `< 0`.

In [ ]:
fs = 28
figsize = (9,6)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

deltaMSE_all_fullbiased = []
nED_full_all_fullbiased = []
nED_cut_all_fullbiased = []
for decay_exp_name in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_fullbiased.append(deltaMSE_mean)
    nED_full_all_fullbiased.append(nED_full)
    nED_cut_all_fullbiased.append(nED_cut)
nED_full_fullbiased = np.concatenate(nED_full_all_fullbiased)
nED_cut_fullbiased = np.concatenate(nED_cut_all_fullbiased)
deltaMSE_all_fullbiased = np.concatenate(deltaMSE_all_fullbiased)
y = deltaMSE_all_fullbiased / norm_y**2.0
x = nED_full_fullbiased - nED_cut_fullbiased
ax.plot(x, y, 'D', color=cmap(0.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy1 = copy.deepcopy(y)

deltaMSE_all_unbiased = []
finalMSE_cut_all_unbiased = []
nED_full_all_unbiased = []
nED_cut_all_unbiased = []
for decay_exp_name in dict_unbiased_all:
    dict_dec = dict_unbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_unbiased.append(deltaMSE_mean)
    nED_full_all_unbiased.append(nED_full)
    nED_cut_all_unbiased.append(nED_cut)
nED_full_unbiased = np.concatenate(nED_full_all_unbiased)
nED_cut_unbiased = np.concatenate(nED_cut_all_unbiased)
deltaMSE_all_unbiased = np.concatenate(deltaMSE_all_unbiased)
y = deltaMSE_all_unbiased / norm_y**2.0
x = nED_full_unbiased - nED_cut_unbiased
ax.plot(x, y, 'D', color=cmap(1.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy2 = copy.deepcopy(y)

xx = np.arange(0.0, 0.06, 0.001)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

In [ ]:
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_biased_and_unbiased.pdf', bbox_inches='tight')

Diagnostic check of the saturating log-compression transform `log(4000/norm_y^2) - log(4000/norm_y^2 - Delta_{f-c}MSE_min)` applied to the pooled `Delta_{f-c}MSE_min` values, used to compress the y-axis range in the next plot.

In [ ]:
yy = np.concatenate((yy1, yy2))
III = np.argsort(yy)
yy = yy[III] / norm_y**2.0

fs = 24
figsize = (10,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

print(np.max(yy))
#ax.plot(yy, yy, 'o', color='tab:blue', markersize=3)
ax.plot(yy, np.log(4000 / norm_y**2.0) - np.log(4000 / norm_y**2.0 - yy), 'o', color='tab:orange', markersize=3)
ax.hlines(0.0, ax.get_xlim()[0], ax.get_xlim()[1], linestyles='dashed', colors='red', linewidth=2)
ax.vlines(0.0, ax.get_ylim()[0], ax.get_ylim()[1], linestyles='dashed', colors='red', linewidth=2)

Repeats the biased-vs-unbiased `Delta_{f-c}MSE_min` vs. effective-dimension-gap scatter with the same log-compressed y-axis transform, to better visualize the sign change and spread across both regimes.

In [ ]:
fs = 24
figsize = (10,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

deltaMSE_all_fullbiased = []
nED_full_all_fullbiased = []
nED_cut_all_fullbiased = []
for decay_exp_name in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_fullbiased.append(deltaMSE_mean)
    nED_full_all_fullbiased.append(nED_full)
    nED_cut_all_fullbiased.append(nED_cut)
nED_full_fullbiased = np.concatenate(nED_full_all_fullbiased)
nED_cut_fullbiased = np.concatenate(nED_cut_all_fullbiased)
deltaMSE_all_fullbiased = np.concatenate(deltaMSE_all_fullbiased)
y = deltaMSE_all_fullbiased / norm_y**2.0
x = nED_full_fullbiased - nED_cut_fullbiased
ax.plot(x, np.log(4000) - np.log(4000 - y), 'o', color=cmap(0.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy1 = copy.deepcopy(y)

deltaMSE_all_unbiased = []
finalMSE_cut_all_unbiased = []
nED_full_all_unbiased = []
nED_cut_all_unbiased = []
for decay_exp_name in dict_unbiased_all:
    dict_dec = dict_unbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    deltaMSE_all_unbiased.append(deltaMSE_mean)
    nED_full_all_unbiased.append(nED_full)
    nED_cut_all_unbiased.append(nED_cut)
nED_full_unbiased = np.concatenate(nED_full_all_unbiased)
nED_cut_unbiased = np.concatenate(nED_cut_all_unbiased)
deltaMSE_all_unbiased = np.concatenate(deltaMSE_all_unbiased)
y = deltaMSE_all_unbiased / norm_y**2.0
x = nED_full_unbiased - nED_cut_unbiased
ax.plot(x, np.log(4000) - np.log(4000 - y), 'o', color=cmap(1.0), markersize=7, markeredgewidth=1.0, markeredgecolor='k')
yy2 = copy.deepcopy(y)

xx = np.arange(0.0, 0.06, 0.001)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)
#ax.set_ylabel(r'$\Delta_{\mathrm{MSE}}$', fontsize=fs)
ax.set_ylabel(r'$\log\frac{4000}{4000-\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

Combines all three bias regimes (fully-biased, unbiased and the `eps`-scanned slightly-biased case) into one `Delta_{f-c}MSE_min` vs. effective-dimension-gap scatter, color-coding each slightly-biased point by its estimated model bias, to show how partial bias interpolates between the biased and unbiased extremes.

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

finalMSEdiff_means_fullbiased = []
nEDdiff_means_fullbiased = []
for decay_exp_name in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    MSE_full_mean = np.asarray(dict_dec['MSE_full_mean'])
    MSE_cut_mean = np.asarray(dict_dec['MSE_cut_mean'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    min_MSE_full_mean = np.min(MSE_full_mean, axis=1)
    min_MSE_cut_mean = np.min(MSE_cut_mean, axis=1)
    diff_minMSE = deltaMSE_mean
    diff_nED = nED_full - nED_cut
    finalMSEdiff_means_fullbiased.append(np.mean(diff_minMSE))
    nEDdiff_means_fullbiased.append(np.mean(diff_nED))
finalMSEdiff_means_fullbiased = np.asarray(finalMSEdiff_means_fullbiased) / norm_y**2.0
nEDdiff_means_fullbiased = np.asarray(nEDdiff_means_fullbiased)

finalMSEdiff_means_unbiased = []
nEDdiff_means_unbiased = []
for decay_exp_name in dict_fullbiased_all:
    dict_dec = dict_fullbiased_all[decay_exp_name]
    nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
    nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
    MSE_full_mean = np.asarray(dict_dec['MSE_full_mean'])
    MSE_cut_mean = np.asarray(dict_dec['MSE_cut_mean'])
    deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
    min_MSE_full_mean = np.min(MSE_full_mean, axis=1)
    min_MSE_cut_mean = np.min(MSE_cut_mean, axis=1)
    diff_minMSE = deltaMSE_mean
    diff_nED = nED_full - nED_cut
    finalMSEdiff_means_unbiased.append(np.mean(diff_minMSE))
    nEDdiff_means_unbiased.append(np.mean(diff_nED))
finalMSEdiff_means_unbiased = np.asarray(finalMSEdiff_means_unbiased) / norm_y**2.0
nEDdiff_means_unbiased = np.asarray(nEDdiff_means_unbiased)

min_bias = 1000.0
max_bias = 0.0
dict_to_plot = dict()
for eps in dict_slightbiased_all:
    dict_eps = dict_slightbiased_all[eps]
    biases_means = []
    finalMSEdiff_means = []
    nEDdiff_means = []
    for decay_exp_name in dict_eps:
        dict_dec = dict_eps[decay_exp_name]
        nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
        nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
        MSE_full_mean = np.asarray(dict_dec['MSE_full_mean'])
        MSE_cut_mean = np.asarray(dict_dec['MSE_cut_mean'])
        bias_full = np.asarray(dict_dec['model_bias_full'])
        deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
        min_MSE_full_mean = np.min(MSE_full_mean, axis=1)
        min_MSE_cut_mean = np.min(MSE_cut_mean, axis=1)
        diff_minMSE = deltaMSE_mean
        diff_nED = nED_full - nED_cut
        finalMSEdiff_means.append(np.mean(diff_minMSE))
        nEDdiff_means.append(np.mean(diff_nED))
        biases_means.append(np.mean(bias_full))
    finalMSEdiff_means = np.asarray(finalMSEdiff_means)
    nEDdiff_means = np.asarray(nEDdiff_means)
    biases_means = np.asarray(biases_means)
    min_bias_i = np.min(biases_means)
    if min_bias_i<min_bias:
        min_bias = min_bias_i
    max_bias_i = np.max(biases_means)
    if max_bias_i>max_bias:
        max_bias = max_bias_i
    dict_eps = dict()
    dict_eps['biases'] = biases_means / norm_y
    dict_eps['nED_diffs'] = nEDdiff_means
    dict_eps['MSE_diffs'] = finalMSEdiff_means / norm_y**2.0
    dict_to_plot[eps] = dict_eps

#cmap = matplotlib.colormaps['cividis']
#log_min_bias = np.log10(min_bias)
#log_max_bias = np.log10(max_bias)
#def map_z_to_cmap(z):
#    log_z = np.log10(z)
#    log_z_t = (log_z - log_min_bias) / (log_max_bias - log_min_bias)
#    return cmap(log_z_t)
#
#for nep in range(len(eps_vec)):
#    eps = eps_vec[nep]
#    name_eps = name_eps_vec[nep]
#    dict_eps = dict_to_plot[eps]
#    y = dict_eps['MSE_diffs']
#    x = dict_eps['nED_diffs']
#    z = dict_eps['biases']
#    for i in range(len(x)):
#        clr = map_z_to_cmap(z[i])
#        ax.plot(x[i], y[i], 'o', color=clr, markersize=7)


cmap = matplotlib.colormaps['cividis']
xx = []
yy = []
zz = []
for nep in range(len(eps_vec)):
    eps = eps_vec[nep]
    name_eps = name_eps_vec[nep]
    dict_eps = dict_to_plot[eps]
    y = dict_eps['MSE_diffs']
    x = dict_eps['nED_diffs']
    z = dict_eps['biases']
    for i in range(len(x)):
        xx.append(x[i])
        yy.append(y[i])
        zz.append(z[i])
xx = np.asarray(xx)
yy = np.asarray(yy)
zz = np.asarray(zz)

fA = ax.scatter(xx, yy, marker='o', s=70, c=zz, cmap=cmap)
cbar = fig.colorbar(fA, ax=ax, extend='both')
cbar.ax.set_ylabel(r'$\delta_{\mathrm{data}}$', fontsize=fs)

xx = np.arange(0.0, 0.06, 0.001)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)

#ax1 = fig.add_axes([0.91, 0.12, 0.03, 0.75])
#cb = matplotlib.colorbar.ColorbarBase(ax1, cmap=cmap, orientation='vertical')

#ax.legend(fontsize=22)
ax.set_xlabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

In [ ]:
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_allbiases.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_vs_DeltaEffDim_allbiases.pdf', bbox_inches='tight')

`Delta_{f-c}MSE_min` plotted against the partial-bias parameter `delta_data` (`eps`, log-scaled x-axis), averaged over model draws for each `decay_exp`, color-coded by the corresponding effective-dimension gap, showing the sign of `Delta_{f-c}MSE_min` flipping as bias is turned on/off.

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

dict_to_plot = dict()
for decay_exp_name in decay_exp_name_vec:
    nEDdiff_means = []
    diffs_mses = []
    diffs_neds = []
    biases = []
    for eps in eps_vec:
        dict_eps = dict_slightbiased_all[eps]
        dict_dec = dict_eps[decay_exp_name]
        nED_full = np.asarray(dict_dec['norm_eff_dim_full'])
        nED_cut = np.asarray(dict_dec['norm_eff_dim_cut'])
        MSE_full_mean = np.asarray(dict_dec['MSE_full_mean'])
        MSE_cut_mean = np.asarray(dict_dec['MSE_cut_mean'])
        deltaMSE_mean = np.asarray(dict_dec['delta_minMSE_mean'])
        bias_full = np.asarray(dict_dec['model_bias_full'])
        min_MSE_full_mean = np.min(MSE_full_mean, axis=1)
        min_MSE_cut_mean = np.min(MSE_cut_mean, axis=1)
        diff_minMSE = deltaMSE_mean
        diff_nED = nED_full - nED_cut
        diffs_mses.append(np.mean(diff_minMSE))
        diffs_neds.append(np.mean(diff_nED))
        biases.append(np.mean(bias_full))
    diffs_mses = np.asarray(diffs_mses)
    diffs_neds = np.asarray(diffs_neds)
    biases = np.asarray(biases)
    nEDdiff_means.append(np.mean(diffs_neds))
    dict_exp = dict()
    dict_exp['biases'] = biases / norm_y
    dict_exp['diffs_mses'] = diffs_mses / norm_y**2.0
    dict_exp['diffs_neds'] = diffs_neds
    dict_exp['nEDdiff_means'] = nEDdiff_means
    dict_to_plot[decay_exp_name] = dict_exp

#cmap = matplotlib.colormaps['viridis']
#for decay_exp_name in dict_to_plot:
#    dict_exp = dict_to_plot[decay_exp_name]
#    y = dict_exp['diffs_mses']
#    x = dict_exp['biases']
#    z = dict_exp['diffs_neds']
#    for i in range(len(x)):
#        clr = cmap((z[i] - 0.01)/(0.06 - 0.01))
#        ax.plot(x[i], y[i], 'o', color=clr, markersize=7)

cmap = matplotlib.colormaps['viridis']
xx = []
yy = []
zz = []
for decay_exp_name in dict_to_plot:
    dict_exp = dict_to_plot[decay_exp_name]
    y = dict_exp['diffs_mses']
    x = dict_exp['biases']
    z = dict_exp['diffs_neds']
    for i in range(len(x)):
        xx.append(x[i])
        yy.append(y[i])
        zz.append(z[i])
xx = np.asarray(xx)
yy = np.asarray(yy)
zz = np.asarray(zz)

fA = ax.scatter(xx, yy, marker='o', s=70, c=zz, cmap=cmap)
cbar = fig.colorbar(fA, ax=ax, extend='both')
cbar.ax.set_ylabel(r'$\hat{d}^{\mathrm{full}}_{\mathrm{eff}}-\hat{d}^{\mathrm{cut}}_{\mathrm{eff}}$', fontsize=fs)

xx = np.arange(np.min(xx), np.max(xx), 0.1)
ax.plot(xx, np.zeros(len(xx)), '--', color='red', linewidth=3)


#ax.legend(fontsize=22)
ax.set_xlabel(r'$\delta_{\mathrm{data}}$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

In [ ]:
fig.savefig('DeltaMSEmin_vs_deltaData_allEDs.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_vs_deltaData_allEDs.pdf', bbox_inches='tight')